# Lab-2

In [1]:
import pandas as pd
import numpy as np
import torch as t
import torch.nn as nn

## 1. Use the dataset in salary.zip to create a linear regression model. Split the dataset into  training and test using a 70: 30 split. Find the Mean Squared Error on the test set. If you  shuffle the data and then split using the mentioned splits, does it have an impact in the  model? Show it by shuffling the data 5 times. 

In [2]:
DF = pd.read_csv('Salary_dataset.csv', index_col = 0)
DF

,YearsExperience,Salary
0,1.2,39344.0
1,1.4,46206.0
2,1.6,37732.0
3,2.1,43526.0
4,2.3,39892.0
5,3.0,56643.0
6,3.1,60151.0
7,3.3,54446.0
8,3.3,64446.0
9,3.8,57190.0


In [36]:
def Shuffle_And_Split(DF, split: int):
    Shuffle = DF.sample(frac=1).reset_index(drop=True)  
    split = int(split*len(DF))
    Train = Shuffle.iloc[:split, :]
    Test = Shuffle.iloc[split:, :]
    return Train, Test

In [4]:
class LinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.weight = nn.Parameter(t.randn(1, requires_grad = True, dtype = t.float))
        self.bias = nn.Parameter(t.randn(1, requires_grad = True, dtype = t.float))
    
    def forward(self,x):
        y = self.weight * x + self.bias
        return y

In [62]:
def Test(model, X: t.Tensor):
    model.eval()
    with t.inference_mode():
        y_preds = model(X)   # ← no forward() call
        return y_preds

In [92]:
def Train(model, epoch, X_train, X_test, y_train, y_test, LossFunction, Optimizer):
    step = max(1, epoch // 10)

    # Initial training error
    y_preds = Test(model, X_train)
    err = LossFunction(y_preds, y_train)
    print(f'Initial Training Error: {err.item():.4f}')

    for e in range(1, epoch + 1):
        model.train()

        y_preds = model(X_train)
        loss = LossFunction(y_preds, y_train)

        Optimizer.zero_grad()
        loss.backward()
        t.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        Optimizer.step()

        # Re-evaluate (as you intended)
        y_preds = Test(model, X_train)
        err = LossFunction(y_preds, y_train)
        
        if e % step == 0:
            print(f'Epoch: {e} - Error: {err.item():.4f}')

In [190]:
X_Train, X_Test = Shuffle_And_Split(DF, 0.7)

In [251]:
def Run(shuf: int, DF, epoch: int = 1000, split: float = 0.7):
 for _ in range(shuf):
    X_Train, X_Test = Shuffle_And_Split(DF, split)

    y_Train = X_Train.iloc[:, 1].to_numpy(dtype=float)
    X_Train = X_Train.iloc[:, 0].to_numpy(dtype=float)
    
    y_Test = X_Test.iloc[:, 1].to_numpy(dtype=float)
    X_Test = X_Test.iloc[:, 0].to_numpy(dtype=float)

    Mx = X_Test.mean()
    Sx = X_Test.std()
    
    My = y_Test.mean()
    Sy = y_Test.std()

    X_Train = (X_Train - X_Train.mean()) / X_Train.std()
    X_Test  = (X_Test  - X_Test.mean())  / X_Test.std()
    
    #y_Train = (y_Train - y_Train.mean()) / y_Train.std()
    #y_Test  = (y_Test  - y_Test.mean())  / y_Test.std()

    X_Train = t.tensor(X_Train, dtype=t.float32)
    y_Train = t.tensor(y_Train, dtype=t.float32)
    
    X_Test = t.tensor(X_Test, dtype=t.float32)
    y_Test = t.tensor(y_Test, dtype=t.float32)

    model = LinearRegression()
    LossFunction = nn.MSELoss()
    Optimizer = t.optim.SGD(params = model.parameters(), lr=1e-2)
    Train(model, epoch, X_Train, X_Test, y_Train, y_Test, LossFunction, Optimizer)

    Yt_numpy = y_Test.numpy()
    Y_pred = Test(model, X_Test).numpy()
    l = 0.5*((Yt_numpy - Y_pred)**2)
    l_sum = sum(l)/len(Yt_numpy)
    print(f'Loss: {l_sum}\n')

In [191]:
y_Train = X_Train.iloc[:, 1].to_numpy(dtype=float)
X_Train = X_Train.iloc[:, 0].to_numpy(dtype=float)

y_Test = X_Test.iloc[:, 1].to_numpy(dtype=float)
X_Test = X_Test.iloc[:, 0].to_numpy(dtype=float)

In [192]:
Mx = X_Test.mean()
Sx = X_Test.std()

My = y_Test.mean()
Sy = y_Test.std()

In [193]:
X_Train = (X_Train - X_Train.mean()) / X_Train.std()
X_Test  = (X_Test  - X_Test.mean())  / X_Test.std()

y_Train = (y_Train - y_Train.mean()) / y_Train.std()
y_Test  = (y_Test  - y_Test.mean())  / y_Test.std()

In [198]:
X_Train = t.tensor(X_Train, dtype=t.float32)
y_Train = t.tensor(y_Train, dtype=t.float32)

X_Test = t.tensor(X_Test, dtype=t.float32)
y_Test = t.tensor(y_Test, dtype=t.float32)

In [199]:
X_Train

tensor([ 0.4502,  0.8502,  0.5593,  1.1411, -1.5497, -0.9679, -0.2407, -1.4770,
         1.7229, -0.1680, -1.6224, -0.5679,  1.4320,  1.4683, -0.0952,  0.9593,
        -0.8588,  0.1593,  0.1229, -0.9316, -0.3861])

In [201]:
model = LinearRegression()

In [202]:
LossFunction = nn.MSELoss()
Optimizer = t.optim.SGD(params = model.parameters(), lr=1e-2)

In [203]:
Train(model, 1000, X_Train, X_Test, y_Train, y_Test, LossFunction, Optimizer)

Initial Training Error: 3.7356
Epoch: 100 - Error: 0.8919
Epoch: 200 - Error: 0.0663
Epoch: 300 - Error: 0.0425
Epoch: 400 - Error: 0.0421
Epoch: 500 - Error: 0.0421
Epoch: 600 - Error: 0.0421
Epoch: 700 - Error: 0.0421
Epoch: 800 - Error: 0.0421
Epoch: 900 - Error: 0.0421
Epoch: 1000 - Error: 0.0421


In [223]:
Yt_numpy = y_Test.numpy()
Yt_numpy

array([-0.1688394 , -0.92384344, -0.4534659 ,  2.0798788 ,  1.4553415 ,
       -0.39998344, -0.1217978 , -0.40412402, -1.0631663 ], dtype=float32)

In [224]:
Y_pred = Test(model, X_Test).numpy()

In [225]:
l = 0.5*((Yt_numpy - Y_pred)**2)
l_sum = sum(l)/len(Yt_numpy)
print(f'Loss: {l_sum}')

Loss: 0.016564754769206047


In [210]:
Yt_numpy = Yt_numpy*float(Sy) + float(My)

In [211]:
Yt_numpy

array([ 63219.   ,  43526.   ,  55795.   , 121873.   , 105583.   ,
        57190.   ,  64446.   ,  57082.   ,  39892.004], dtype=float32)

In [219]:
Y_pred*float(Sy) + float(My)

array([ 60016.543,  42674.07 ,  60929.305, 120258.82 , 106567.39 ,
        58191.02 ,  53627.21 ,  61842.066,  44499.594], dtype=float32)

In [252]:
Run(5, DF)

Initial Training Error: 6312325632.0000
Epoch: 100 - Error: 6312167936.0000
Epoch: 200 - Error: 6312009216.0000
Epoch: 300 - Error: 6311850496.0000
Epoch: 400 - Error: 6311692288.0000
Epoch: 500 - Error: 6311533568.0000
Epoch: 600 - Error: 6311374848.0000
Epoch: 700 - Error: 6311216640.0000
Epoch: 800 - Error: 6311058432.0000
Epoch: 900 - Error: 6310899712.0000
Epoch: 1000 - Error: 6310740480.0000
Loss: 3473680128.0

Initial Training Error: 6175802880.0000
Epoch: 100 - Error: 6175646720.0000
Epoch: 200 - Error: 6175489024.0000
Epoch: 300 - Error: 6175332864.0000
Epoch: 400 - Error: 6175175680.0000
Epoch: 500 - Error: 6175019008.0000
Epoch: 600 - Error: 6174862336.0000
Epoch: 700 - Error: 6174704640.0000
Epoch: 800 - Error: 6174548480.0000
Epoch: 900 - Error: 6174391296.0000
Epoch: 1000 - Error: 6174234624.0000
Loss: 3632782336.0

Initial Training Error: 6488868352.0000
Epoch: 100 - Error: 6488707584.0000
Epoch: 200 - Error: 6488546816.0000
Epoch: 300 - Error: 6488386560.0000
Epoch: 400

## 2. Use the dataset in salary.zip to create a linear regression model. Split the dataset into training and test using a 80: 20 split. Find the Mean Squared Error on the test set. As the dataset consists of different scales (units) for different columns, does normalization play a factor in the performance of the model? Use min-max scaling and standard normalization to show the impact of normalization.

In [253]:
X_Train, X_Test = Shuffle_And_Split(DF, 0.8)

y_train = X_Train.iloc[:, 1].to_numpy(dtype=float)
X_train = X_Train.iloc[:, 0].to_numpy(dtype=float)
    
y_test = X_Test.iloc[:, 1].to_numpy(dtype=float)
X_test = X_Test.iloc[:, 0].to_numpy(dtype=float)

In [254]:
X_Test.shape
X_Train.shape

(24, 2)

In [259]:
def mean_squared_error(L1, L2):
    l = 0.5*((L1 - L2)**2)
    l_sum = sum(l)/len(L1)
    return l_sum

In [255]:
X_train_mm = (X_train - X_train_min) / (X_train_max - X_train_min)
X_test_mm  = (X_test - X_train_min) / (X_train_max - X_train_min)

# Convert to PyTorch tensors (2D)
X_train_mm_t = t.tensor(X_train_mm.reshape(-1,1), dtype=t.float32)
X_test_mm_t  = t.tensor(X_test_mm.reshape(-1,1), dtype=t.float32)
y_train_t    = t.tensor(y_train.reshape(-1,1), dtype=t.float32)
y_test_t     = t.tensor(y_test.reshape(-1,1), dtype=t.float32)

In [260]:
model = LinearRegression()
loss_fn = nn.MSELoss()
optimizer = t.optim.SGD(model.parameters(), lr=0.05)

epochs = 1000
for e in range(1, epochs+1):
    model.train()
    y_pred = model(X_train_mm_t)
    loss = loss_fn(y_pred, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if e % (epochs//10) == 0:
        print(f"Epoch {e}, Loss: {loss.item():.4f}")

model.eval()
with t.inference_mode():
    y_pred_test = model(X_test_mm_t)
    mse = mean_squared_error(y_test_t.numpy(), y_pred_test.numpy())
print("Test MSE (Min-Max features):", mse)

Epoch 100, Loss: 93899648.0000
Epoch 200, Loss: 44727284.0000
Epoch 300, Loss: 32921288.0000
Epoch 400, Loss: 30086760.0000
Epoch 500, Loss: 29406208.0000
Epoch 600, Loss: 29242814.0000
Epoch 700, Loss: 29203586.0000
Epoch 800, Loss: 29194166.0000
Epoch 900, Loss: 29191910.0000
Epoch 1000, Loss: 29191370.0000
Test MSE (Min-Max features): [21471090.]


In [256]:
X_train_mean = X_train.mean()
X_train_std  = X_train.std()

X_train_std_scaled = (X_train - X_train_mean) / X_train_std
X_test_std_scaled  = (X_test - X_train_mean) / X_train_std

# Convert to PyTorch tensors (2D)
X_train_std_t = t.tensor(X_train_std_scaled.reshape(-1,1), dtype=t.float32)
X_test_std_t  = t.tensor(X_test_std_scaled.reshape(-1,1), dtype=t.float32)
# Labels remain the same
y_train_t = t.tensor(y_train.reshape(-1,1), dtype=t.float32)
y_test_t  = t.tensor(y_test.reshape(-1,1), dtype=t.float32)

In [264]:
model = LinearRegression()
loss_fn = nn.MSELoss()
optimizer = t.optim.SGD(model.parameters(), lr=0.05)

epochs = 1000
for e in range(1, epochs+1):
    model.train()
    y_pred = model(X_train_std_t)
    loss = loss_fn(y_pred, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if e % (epochs//10) == 0:
        print(f"Epoch {e}, Loss: {loss.item():.4f}")

model.eval()
with t.inference_mode():
    y_pred_test = model(X_test_std_t)
    mse = mean_squared_error(y_test_t.numpy(), y_pred_test.numpy())
print("Test MSE (Norm. features):", mse)

Epoch 100, Loss: 29191194.0000
Epoch 200, Loss: 29191190.0000
Epoch 300, Loss: 29191190.0000
Epoch 400, Loss: 29191190.0000
Epoch 500, Loss: 29191190.0000
Epoch 600, Loss: 29191190.0000
Epoch 700, Loss: 29191190.0000
Epoch 800, Loss: 29191190.0000
Epoch 900, Loss: 29191190.0000
Epoch 1000, Loss: 29191190.0000
Test MSE (Norm. features): [21437218.]


In [265]:
X_train_none = t.tensor(X_train.reshape(-1,1), dtype=t.float32)
X_test_none  = t.tensor(X_test.reshape(-1,1), dtype=t.float32)

In [267]:
model = LinearRegression()
loss_fn = nn.MSELoss()
optimizer = t.optim.SGD(model.parameters(), lr=0.01)

epochs = 1000
for e in range(1, epochs+1):
    model.train()
    y_pred = model(X_train_none)
    loss = loss_fn(y_pred, y_train_t)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if e % (epochs//10) == 0:
        print(f"Epoch {e}, Loss: {loss.item():.4f}")

model.eval()
with t.inference_mode():
    y_pred_test = model(X_test_none)
    mse = mean_squared_error(y_test_t.numpy(), y_pred_test.numpy())
print("Test MSE:", mse)

Epoch 100, Loss: 80771848.0000
Epoch 200, Loss: 52774500.0000
Epoch 300, Loss: 39973788.0000
Epoch 400, Loss: 34121132.0000
Epoch 500, Loss: 31445222.0000
Epoch 600, Loss: 30221768.0000
Epoch 700, Loss: 29662374.0000
Epoch 800, Loss: 29406614.0000
Epoch 900, Loss: 29289686.0000
Epoch 1000, Loss: 29236218.0000
Test MSE: [21149228.]
